In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.api import VAR
import statsmodels.api as sm
from scipy import stats as sp_stats

For the private starts we take Started - Private Enterprises.

For social starts we sum Started - Housing Associations and Started - Local Authorities

In [ ]:
# CONFIG
IN = "../../data"

df = pd.read_excel(f"{IN}/raw/starts/indicatorsofukhousebuilding.xlsx",
                   sheet_name="1b",
                   skiprows=5)

# Replacing ONS placeholder with low number - 5
df["Started - Local Authorities"] = df["Started - Local Authorities"].replace("[low]", 5)
df["Started - Local Authorities"] = pd.to_numeric(df["Started - Local Authorities"], errors="coerce")
df["Started - Housing Associations"] = pd.to_numeric(df["Started - Housing Associations"], errors="coerce")


df["lhsoc"] = np.log(df["Started - Housing Associations"] + df["Started - Local Authorities"])
df["lhstarts"] = np.log(df["Started - Private Enterprise"])

cols_to_keep = ["Period", "lhstarts", "lhsoc"]

starts = df[cols_to_keep].copy()

starts = starts.set_index("Period")

# Mapping quarters
quarter_map = {
    "Jan - Mar": "Q1",
    "Apr - Jun": "Q2",
    "Jul - Sep": "Q3",
    "Oct - Dec": "Q4"
}

# Clean up index
idx_str = starts.index.astype(str).str.strip()

# Extract the month range text and the 4-digit year
months = idx_str.str.extract(r'(Jan - Mar|Apr - Jun|Jul - Sep|Oct - Dec)')[0]
years = idx_str.str.extract(r'(\d{4})')[0]

# Translate the months into Q1, Q2, etc.
quarters = months.map(quarter_map)

# Glue into strings
quarter_strings = years + quarters

# Convert to Period objects
starts.index = pd.PeriodIndex(quarter_strings, freq='Q')

# Naming index
starts.index.name = 'Date'

We confirm I(1) - I should test trend stationarity as well

In [ ]:
adf_stat, adf_p, *_ = adfuller(starts["lhsoc"], autolag="AIC")
kpss_stat, kpss_p, *_ = kpss(starts["lhsoc"], regression="c", nlags="auto")

print("Log Level Test")
print("ADF: ", adf_stat, adf_p)
print("KPSS: ", kpss_stat, kpss_p)

starts["dlhsoc"] = starts["lhsoc"].diff()
starts["dlhstarts"] = starts["lhstarts"].diff()

adf_stat, adf_p, *_ = adfuller(starts["dlhsoc"].dropna(), autolag="AIC")
kpss_stat, kpss_p, *_ = kpss(starts["dlhsoc"].dropna(), regression="c", nlags="auto")

print(f"\nDifference Log Test")
print("ADF: ", adf_stat, adf_p)
print("KPSS: ", kpss_stat, kpss_p)

Remember we found that private starts is I(0) so we need to specify with constant and trend

In [ ]:
var_data = starts[["lhstarts", "lhsoc"]].dropna()

model = VAR(var_data)
lag_order_results = model.select_order(maxlags=8, trend="ct")  
# print(lag_order_results.summary())

p = 5

# Todo Yamamoto Refit
d_max = 1
p_aug = p + d_max # 6

ty_model = VAR(var_data)
ty_results = ty_model.fit(p_aug, trend="ct")
print(ty_results.summary())

# Residual autocorrelation test
print(ty_results.test_whiteness(nlags=8))


Todo Yamamoto Wald Test for Granger Casuality - Bidirectional

In [ ]:
def ty_granger_wald(results, p, cause, effect):
    """
    Toda-Yamamoto Granger non-causality Wald test.
    Tests H0: 'cause' does not Granger-cause 'effect'.
    """
    cov_full = results.cov_params()
    mi = cov_full.index  # MultiIndex (coef_name, equation)

    coef_names = [f"L{lag}.{cause}" for lag in range(1, p + 1)]
    target_tuples = [(name, effect) for name in coef_names]

    # Positions in the MultiIndex
    positions = [mi.get_loc(t) for t in target_tuples]

    # Coefficient values for the effect equation
    beta_eq = results.params[effect]  # Series indexed by coef name
    beta_restricted = beta_eq.loc[coef_names].values

    cov_vals = cov_full.values

    R = np.zeros((len(positions), cov_vals.shape[0]))
    for i, pos in enumerate(positions):
        R[i, pos] = 1

    middle = R @ cov_vals @ R.T
    wald_stat = beta_restricted.T @ np.linalg.inv(middle) @ beta_restricted
    df = len(coef_names)
    p_value = 1 - sp_stats.chi2.cdf(wald_stat, df)

    return wald_stat, df, p_value

p = 5
wald_soc_to_starts, df1, pval1 = ty_granger_wald(ty_results, p, cause="lhsoc", effect="lhstarts")
wald_starts_to_soc, df2, pval2 = ty_granger_wald(ty_results, p, cause="lhstarts", effect="lhsoc")

print(f"lhsoc -> lhstarts:  Wald = {wald_soc_to_starts:.3f}, df = {df1}, p = {pval1:.4f}")
print(f"lhstarts -> lhsoc:  Wald = {wald_starts_to_soc:.3f}, df = {df2}, p = {pval2:.4f}")

Controlled Regression Pre-Processing

In [ ]:
controls = pd.read_csv(f"{IN}/python_master/england_master.csv")

# Filter to 1978Q1 and onwards

controls["Unnamed: 0"] = pd.PeriodIndex(controls["Unnamed: 0"], freq="Q")
controls = controls.rename(columns={"Unnamed: 0": "Date"})
controls = controls.set_index("Date")

# Filter to 1978Q1 onward
controls = controls[controls.index >= pd.Period("1978Q1", freq="Q")]

controls = controls.drop(columns=["starts", "p_def"])

# Tansformations
controls["lrprc"] = np.log(controls["hprice"] / controls["gdp_def"])    # real house prices
controls["lrcc"] = np.log(controls["cc"] / controls["gdp_def"])          # real construction costs
controls["lvol"] = np.log(controls["vol"])                         # transactions volume
controls["lstock"] = np.log(controls["hstock"])                    # housing stock
controls["r3"] = controls["rate"]                                   # interest rate



df = controls.join(starts, how="inner")

cols_to_keep = [
    "lhstarts", "lhsoc", "dlhstarts", "dlhsoc",
    "lrprc", "lvol", "r3", "lstock", "lrcc"
]

master = df[cols_to_keep].copy()
master = master.dropna()

Contemporaneous Regression - Unrestricted ECM

In [ ]:
m = master.copy()

for L in [1, 2, 3, 4]:
    m[f"dlhstarts_L{L}"] = m["dlhstarts"].shift(L)
    m[f"dlhsoc_L{L}"] = m["dlhsoc"].shift(L)
for c in ["lhstarts", "lrprc", "lvol", "r3", "lrcc"]:
    m[f"{c}_L1"] = m[c].shift(1)

for q in [2, 3, 4]:
    m[f"q{q}"] = (m.index.quarter == q).astype(float)
# Crisis dummies
for d in ["08Q3", "20Q2", "20Q3", "23Q2"]:
    m[f"d{d}"] = (m.index == pd.Period(f"20{d}", freq="Q")).astype(float)

m = m.dropna()

soc = ["dlhsoc", "dlhsoc_L1", "dlhsoc_L2", "dlhsoc_L3", "dlhsoc_L4"]
X = m[["lhstarts_L1", "lrprc_L1", "lvol_L1", "r3_L1", "lrcc_L1",
       "dlhstarts_L1", "dlhstarts_L2", "dlhstarts_L3", "dlhstarts_L4",
       "q2", "q3", "q4", "d08Q3", "d20Q2", "d20Q3", "d23Q2"] + soc]
X = sm.add_constant(X)

res = sm.OLS(m["dlhstarts"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(res.summary())
print(res.f_test("q2 = q3 = q4 = 0"))

# DIAGNOSTICS

In [ ]:
alpha = res.params["lhstarts_L1"]
lag_sum = " + ".join(soc[1:]) + " = 0"
all_sum = " + ".join(soc) + " = 0"

# 1. IS THIS A VALID ECM? not a result, a licence to interpret the rest
print("=== 1. specification validity ===")
print(f"alpha {alpha:.3f}  t {res.tvalues['lhstarts_L1']:.2f}  "
      f"half-life {np.log(0.5) / np.log(1 + alpha):.2f}q   (need -1 < alpha < 0)")
print("BG LM, p:", acorr_breusch_godfrey(res, nlags=4)[:2], "(replaces DW)")

# 2. MECHANISM: same-quarter S106 linkage
print("\n=== 2. same-quarter co-movement ===")
print(f"dlhsoc {res.params['dlhsoc']:.3f}  z {res.tvalues['dlhsoc']:.2f}")

# 3. THE CROWDING-OUT TEST: lagged block, contemporaneous controlled
print("\n=== 3. crowding out ===")
print(res.t_test(lag_sum))                                   # cumulative -> report CI as bound
print(res.f_test(", ".join(f"{v} = 0" for v in soc[1:])))     # joint -> rules out offsetting lags

# 4. CO-MOVEMENT MAGNITUDE: descriptive only
print("\n=== 4. full social block ===")
print(res.t_test(all_sum))

# 5. EXCLUSION ARGUMENT: does the social block absorb the price channel?
print("\n=== 5. long-run vector ===")
X2 = X.drop(columns=soc)
res2 = sm.OLS(m["dlhstarts"], X2).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
for label, r in [("with social", res), ("no social  ", res2)]:
    a = r.params["lhstarts_L1"]
    print(label, f"alpha {a:6.3f} ",
          "  ".join(f"{c} {-r.params[c + '_L1'] / a:6.3f}" for c in ["lrprc", "lvol", "lrcc", "r3"]))

# 6. IS THE NULL A BANDWIDTH ARTEFACT?
print("\n=== 6. HAC bandwidth ===")
for L in [4, 5, 8]:
    t = sm.OLS(m["dlhstarts"], X).fit(cov_type="HAC", cov_kwds={"maxlags": L}).t_test(lag_sum)
    print(f"maxlags {L}: sum {float(t.effect):.3f}  t {float(t.tvalue):.2f}  p {float(t.pvalue):.3f}")